## RUN GAME

In [ ]:
# TODO: Realize o pré processamento aqui
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Carregar o dataset definindo os nomes das colunas (conforme tic-tac-toe.names)
colunas = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'Class']
df = pd.read_csv('tic-tac-toe.data', names=colunas)

# 2. Separar atributos (X) e classe alvo (Y)
X = df.drop('Class', axis=1)
Y = df['Class']

# 3. Transformar variáveis categóricas em numéricas (One-Hot Encoding visto na Aula 03)
X_encoded = pd.get_dummies(X)

# Guardar as colunas originais do treinamento para usar o .reindex() nas novas amostras do jogo
colunas_treinamento = X_encoded.columns

# 4. Dividir em conjuntos de treino e teste
X_train, X_test, Y_train, Y_test = train_test_split(
    X_encoded, Y, test_size=0.2, random_state=42
)

# 5. Normalização dos dados com StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Treinamento do modelo (MLPClassifier estudado na Aula 05)
modelo_ia = MLPClassifier(hidden_layer_sizes=(18, 9), max_iter=2000, random_state=42)
modelo_ia.fit(X_train_scaled, Y_train)

# Avaliação da acurácia do modelo treinado
y_pred = modelo_ia.predict(X_test_scaled)
print(f"Acurácia do modelo no teste: {accuracy_score(Y_test, y_pred) * 100:.2f}%")

Acurácia do modelo no teste: 98.44%


In [ ]:
# Imports
from tic_tac_toe import run
from tic_tac_toe import constants

In [ ]:
def get_ai_move(board, current_player='x'):
    # Mapear os índices numéricos (0 a 8) de volta para as coordenadas ('A1' a 'C3')
    index_to_move = {v: k for k, v in constants.MAPPING.items()}

    # Identificar todas as posições vazias (' ') no tabuleiro atual
    posicoes_vazias = [i for i, casa in enumerate(board) if casa == ' ']

    melhor_jogada = None
    maior_probabilidade = -1.0

    # Descobrir em qual coluna do predict_proba está a classe 'positive'
    indice_positivo = list(modelo_ia.classes_).index('positive')

    # Testar cada jogada possível nas casas vazias
    for pos in posicoes_vazias:
        # Criar cópia do tabuleiro para simular a jogada
        tabuleiro_simulado = board.copy()
        tabuleiro_simulado[pos] = current_player

        # Converter o tabuleiro do jogo ('X', 'O', ' ') para o padrão do dataset ('x', 'o', 'b')
        # Como o dataset foi treinado assumindo 'x' como vencedor ('positive'),
        # espelhamos os símbolos caso a IA esteja jogando como 'O' para avaliar a vitória dela
        linha_dataset = []
        for casa in tabuleiro_simulado:
            if casa == ' ':
                linha_dataset.append('b')
            elif casa.upper() == current_player.upper():
                linha_dataset.append('x')
            else:
                linha_dataset.append('o')

        # Transformar a jogada simulada em DataFrame e aplicar get_dummies + reindex
        df_jogada = pd.DataFrame([linha_dataset], columns=colunas[:-1])
        df_jogada_encoded = pd.get_dummies(df_jogada)
        df_jogada_encoded = df_jogada_encoded.reindex(columns=colunas_treinamento, fill_value=0)

        # Aplicar a normalização do StandardScaler
        jogada_scaled = scaler.transform(df_jogada_encoded)

        # Obter a probabilidade de a configuração levar à vitória ('positive')
        prob_vitoria = modelo_ia.predict_proba(jogada_scaled)[0][indice_positivo]

        # Escolher a jogada com a maior probabilidade estimada pelo modelo
        if prob_vitoria > maior_probabilidade:
            maior_probabilidade = prob_vitoria
            melhor_jogada = index_to_move[pos]

    print(f"IA ({current_player}) escolheu a posição: {melhor_jogada}")
    return melhor_jogada

In [ ]:
# Initialize a new game board
run.runGame(get_ai_move)

Welcome to Tic Tac Toe!
Choose game mode:
1. Player vs Player
2. Player vs AI
Enter 1 or 2: 2
Your Player 2 is our implemented AI.

   1   2   3
A    |   |  
  ---+---+---
B    |   |  
  ---+---+---
C    |   |  

Enter name for Player 1 (X | O): X
Player 1 is 'X' and Player 2 is 'O'

   1   2   3
A    |   |  
  ---+---+---
B    |   |  
  ---+---+---
C    |   |  

Player X, enter your move (e.g., A1, B2): A1

   1   2   3
A  X |   |  
  ---+---+---
B    |   |  
  ---+---+---
C    |   |  

IA (O) escolheu a posição: A3

   1   2   3
A  X |   | O
  ---+---+---
B    |   |  
  ---+---+---
C    |   |  

Player X, enter your move (e.g., A1, B2): B2

   1   2   3
A  X |   | O
  ---+---+---
B    | X |  
  ---+---+---
C    |   |  

IA (O) escolheu a posição: C3

   1   2   3
A  X |   | O
  ---+---+---
B    | X |  
  ---+---+---
C    |   | O

Player X, enter your move (e.g., A1, B2): B3

   1   2   3
A  X |   | O
  ---+---+---
B    | X | X
  ---+---+---
C    |   | O

IA (O) escolheu a posição: C1